# DRF API Documentation with drf-spectacular

## Why API Documentation?

Without documentation, API consumers must read the source code to find available endpoints, required fields, and response shapes. `drf-spectacular` auto-generates an OpenAPI 3 schema from your DRF views and makes it available as interactive documentation.

Two popular UI renderers:
- **Swagger UI** — interactive, lets you test requests in the browser
- **ReDoc** — clean, read-only, well-suited for public docs


## Installation and Setup

```bash
pip install drf-spectacular
```

```python
# settings.py
INSTALLED_APPS += ['drf_spectacular']

REST_FRAMEWORK = {
    'DEFAULT_SCHEMA_CLASS': 'drf_spectacular.openapi.AutoSchema',
}

SPECTACULAR_SETTINGS = {
    'TITLE':       'My API',
    'DESCRIPTION': 'API for the book store project',
    'VERSION':     '1.0.0',
    'SERVE_INCLUDE_SCHEMA': False,
}
```


## URL Configuration

```python
# urls.py
from drf_spectacular.views import (
    SpectacularAPIView,
    SpectacularSwaggerView,
    SpectacularRedocView,
)

urlpatterns += [
    # Raw OpenAPI schema (JSON or YAML)
    path('api/schema/',         SpectacularAPIView.as_view(),         name='schema'),
    # Swagger UI
    path('api/schema/swagger/', SpectacularSwaggerView.as_view(url_name='schema'), name='swagger-ui'),
    # ReDoc
    path('api/schema/redoc/',   SpectacularRedocView.as_view(url_name='schema'),   name='redoc'),
]
```

Visit `http://127.0.0.1:8000/api/schema/swagger/` to see the interactive docs.


## Generating the Schema File

Export the schema to a static file for use in CI/CD or client SDK generation:

```bash
python manage.py spectacular --color --file schema.yml
```

The schema file can be used with tools such as `openapi-generator` to produce client libraries automatically.


## Customizing with @extend_schema

`@extend_schema` decorates a view to override or enrich the generated documentation.

```python
from drf_spectacular.utils import extend_schema, OpenApiParameter, OpenApiExample
from drf_spectacular.types import OpenApiTypes

class BookViewSet(viewsets.ModelViewSet):
    queryset         = Book.objects.all()
    serializer_class = BookSerializer

    @extend_schema(
        summary='List all books',
        description='Returns a paginated list of books, optionally filtered by author and price range.',
        parameters=[
            OpenApiParameter('author',    OpenApiTypes.STR,   description='Author slug'),
            OpenApiParameter('min_price', OpenApiTypes.FLOAT, description='Minimum price'),
        ],
        responses={200: BookSerializer(many=True)},
        examples=[
            OpenApiExample(
                'Example response',
                value=[{'id': 1, 'title': 'Django for Beginners', 'price': 19.99}],
                response_only=True,
            ),
        ],
    )
    def list(self, request, *args, **kwargs):
        return super().list(request, *args, **kwargs)
```


## Tags, Deprecation, and Auth

### Grouping endpoints with tags
```python
@extend_schema(tags=['Books'])
class BookViewSet(viewsets.ModelViewSet):
    ...
```

Or set a default tag for the whole view via meta:
```python
class BookViewSet(viewsets.ModelViewSet):
    ...
    # drf-spectacular picks up the ViewSet name automatically as a tag
```

### Marking an endpoint as deprecated
```python
@extend_schema(deprecated=True)
def old_endpoint(self, request):
    ...
```

### Security / authentication in the schema
```python
SPECTACULAR_SETTINGS = {
    ...
    'SECURITY': [{'tokenAuth': []}],
    'COMPONENTS': {
        'securitySchemes': {
            'tokenAuth': {
                'type': 'apiKey',
                'in':   'header',
                'name': 'Authorization',
                'description': 'Token-based authentication. Format: `Token <key>`',
            }
        }
    },
}
```


## SerializerMethodField Documentation

`SerializerMethodField` is not introspectable by default. Annotate it with `@extend_schema_field`:

```python
from drf_spectacular.utils import extend_schema_field

class BookSerializer(serializers.ModelSerializer):
    cover_url = serializers.SerializerMethodField()

    @extend_schema_field(serializers.URLField())
    def get_cover_url(self, obj):
        request = self.context.get('request')
        if obj.cover and request:
            return request.build_absolute_uri(obj.cover.url)
        return None
```


## Summary

- `drf-spectacular` auto-generates an OpenAPI 3 schema from DRF views; no manual YAML writing is needed.
- Set `DEFAULT_SCHEMA_CLASS` to `AutoSchema` and add `drf_spectacular` to `INSTALLED_APPS`.
- Expose `SpectacularSwaggerView` and `SpectacularRedocView` for interactive browser-based docs.
- Use `@extend_schema` to add summaries, parameter descriptions, response examples, and tags.
- Export a static schema file with `python manage.py spectacular --file schema.yml` for client SDK generation or CI validation.
- Annotate `SerializerMethodField` with `@extend_schema_field` so the field type appears correctly in the schema.
